In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from ipywidgets import Dropdown, SelectMultiple, interactive_output, VBox, HBox, IntSlider, Layout, Label, HTML
from IPython.display import display, HTML as IPHTML
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [46]:
# load dataset dataset_1980_2020.csv
crime_df = pd.read_csv('dataset_1980_2020.csv')
crime_df.head()
crime_df = crime_df.rename(columns={'Count': 'Year'})

## Visualize Crime Statistics by State
Create a chorpleth map that can be filtered by the various crime statistics and year

In [47]:
def plot_crime_map(df, filters, year=None):
    # filter by year
    if year is not None:
        df = df[df['Year'] == year]

    valid_filters = [f for f in filters if f in df.columns]

    if not valid_filters:
        raise ValueError("no valid filters provided")
    
    # sum accross the selecter filter columns
    df['Filtered Crime Count'] = df[valid_filters].sum(axis=1)
    state_crime = df.groupby('State')['Filtered Crime Count'].sum().reset_index()

    # plot
    fig = px.choropleth(
        state_crime,
        locations='State',
        locationmode='USA-states',
        color='Filtered Crime Count',
        scope="usa",
        color_continuous_scale="Reds",
        labels={'Filtered Crime Count': 'Crime Count'},
        title=f"Crime Map - Filters: {', '.join(filters)}" + (f" ({year})" if year else "")
    )
    fig.show()

def interactive_crime_map(crime_df):
    all_filters = [
        '0 to 11', '12 to 17',
        'Male', 'Female', 'Unknown Gender',
        'White', 'Black', 'Amer. Indian/Alaskan Native', 'Asian/Nat. Hawaiian/Pac Isl', 'Unknown Race',
        'Family', 'Acquaintance', 'Stranger', 'Unknown',
        'Firearm', 'Knife', 'Blunt object', 'Personal', 'Other/unknown Weapon',
        'One offender involved', 'Two or more offenders involved', 'Unknown number of offenders involved'
    ]

    year_options = crime_df['Year'].dropna().unique()
    filter_label = HTML(
        value="<b>Select Filters</b>",
        layout=Layout(margin='0 0 10px 0')
    )  
    year_slider = IntSlider(
        value=int(year_options.min()),
        min=int(year_options.min()),
        max=int(year_options.max()),
        step=1,
        description='Year:',
        continuous_update=False,
        layout=Layout(width='100%', margin='20px 0 0 0')
    )
    year_slider.style = {
        'description_width': '80px',
        'handle_color': '#d62728',
        'font_size': '16px'
    }
    filter_select = SelectMultiple(
        options=all_filters,
        value=('Male',),
        rows=25,
        description = '',
        layout=Layout(
            width='100%',
            height='auto',
            overflow_y='visible',
            white_space='normal'
        )
    )
    filter_select.style = {'description_width': '0px'}


    left_panel = VBox([filter_label, filter_select], layout=Layout(width='300px', height='100%', align_items='stretch'))
    display(IPHTML("<style>.widget-readout { display: none !important; }</style>"))
    
    out = interactive_output(
        lambda filters, year: plot_crime_map(crime_df, list(filters), year),
        {'filters': filter_select, 'year': year_slider}
    )

    right_panel = VBox(
        [out, year_slider],
        layout=Layout(flex='1', height='100%', align_items='stretch')
    )
    full_ui = HBox(
        [left_panel, right_panel],
        layout=Layout(width='100%', height='auto', align_items='flex-start')
    )
    display(full_ui)

interactive_crime_map(crime_df)


## Forecast Future Crime Patterns and Assign Risk Scores ny State

In [48]:
# aggregate offender demographics by state-year
# offender demographics
offender_columns = [
    '0 to 11', '12 to 17',           # Age
    'Male', 'Female', 'Unknown Gender',  # Gender
    'White', 'Black', 'Amer. Indian/Alaskan Native',
    'Asian/Nat. Hawaiian/Pac Isl', 'Unknown Race'  # Race
]

# aggreate by state-year
agg = crime_df.groupby(['State', 'Year'])[offender_columns].sum().reset_index()

# normalize the feature matrix
features = agg[offender_columns]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
features_scaled_df = pd.DataFrame(features_scaled, columns=offender_columns)

# create final modeling df
model_df = pd.concat([agg[['State', 'Year']], features_scaled_df], axis=1)

model_df

,State,Year,0 to 11,12 to 17,Male,Female,Unknown Gender,White,Black,Amer. Indian/Alaskan Native,Asian/Nat. Hawaiian/Pac Isl,Unknown Race
0,AK,1980,-0.349120,-0.518021,-0.511703,-0.638514,-0.106046,-0.422923,-0.619168,2.614647,-0.223448,-0.186376
1,AK,1981,-0.349120,-0.496211,-0.488734,-0.638514,-0.106046,-0.301719,-0.619168,-0.340129,-0.223448,-0.186376
2,AK,1982,-0.349120,-0.365351,-0.396860,-0.011291,-0.106046,-0.180515,-0.619168,4.092034,-0.223448,-0.186376
3,AK,1983,1.462628,-0.518021,-0.488734,-0.638514,-0.106046,-0.382521,-0.619168,2.614647,-0.223448,-0.186376
4,AK,1984,-0.349120,-0.430781,-0.419829,-0.638514,-0.106046,-0.180515,-0.619168,-0.340129,-0.223448,-0.186376
...,...,...,...,...,...,...,...,...,...,...,...,...
1967,WY,2016,-0.349120,-0.561641,-0.557640,-0.638514,-0.106046,-0.422923,-0.619168,-0.340129,-0.223448,-0.186376
1968,WY,2017,-0.349120,-0.561641,-0.557640,-0.638514,-0.106046,-0.422923,-0.619168,-0.340129,-0.223448,-0.186376
1969,WY,2018,-0.349120,-0.561641,-0.557640,-0.638514,-0.106046,-0.422923,-0.619168,-0.340129,-0.223448,-0.186376
1970,WY,2019,-0.349120,-0.561641,-0.557640,-0.638514,-0.106046,-0.422923,-0.619168,-0.340129,-0.223448,-0.186376


In [49]:
# k means clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(features_scaled)
model_df['Pattern'] = cluster_labels


# analyze cluster charactersistics 
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=offender_columns
)

centroids_original = pd.DataFrame(
    scaler.inverse_transform(centroids),
    columns=offender_columns
)

centroids_original.index.name = 'Pattern'

# save pattern assignments for time-series modeling and risk scoring
model_df.to_csv("state_year_pattern.csv", index=False)

centroids_original

,0 to 11,12 to 17,Male,Female,Unknown Gender,White,Black,Amer. Indian/Alaskan Native,Asian/Nat. Hawaiian/Pac Isl,Unknown Race
Pattern,,,,,,,,,,
0,0.036417,9.281136,8.827385,0.785142,4.369993e-03,4.089585,5.163146,0.100510,0.175528,0.075747
1,1.272727,173.106061,161.878788,12.136364,3.787879e-01,84.530303,84.681818,0.348485,3.696970,1.106061
2,1.923077,349.615385,334.923077,16.538462,7.692308e-02,209.461538,115.615385,1.615385,14.692308,10.461538
3,0.474299,47.046729,44.219626,4.121495,9.579439e-02,13.726636,34.000000,0.072430,0.282710,0.348131
4,0.195652,21.010870,19.489130,1.706522,1.387779e-17,9.250000,8.576087,2.619565,0.510870,0.163043


### Analysis of Patterns

#### Pattern 0
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 9 |
| Black | 5 |
| White | 4 |
| 12 to 17 | 9 |

**Interpretation:**
- Low-volume demographic pattern
- Primarily male offenders, slightly more Black than White
- Young age group (12–17) slightly more prevalent
- Possibly small, localized juvenile crimes

#### Pattern 1
| *Feature* | *Notable Values* |
| --- | --- |
| 12 to 17 | 173 |
| Male | 161 |
| White | 85 |
| Black | 85 |

**Interpretation:**
- High youth involvement (12–17)
- High male counts
- Roughly even Black/White representation
- Suggests youth-driven crime, possibly school/community level

#### Pattern 2
| *Feature* | *Notable Values* |
| --- | --- |
| 12 to 17 | 349 |
| Male | 334 |
| White | 209 |
| Black | 115 |

**Interpretation:**
- Extremely high volume
- Majority male youth crimes
- Strongly White-dominated
- Could reflect rural or suburban regions with high reported youth crime

#### Pattern 3
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 44 |
| Black | 34 |
| 12 to 17 | 47 |

**Interpretation:**
- Mid-level youth crime
- Higher representation of Black offenders
- Possibly urban youth crime clusters

#### Pattern 4
| *Feature* | *Notable Values* |
| --- | --- |
| Male | 19 |
| Amer. Indian | 3 |
| 12 to 17 | 21 |

**Interpretation:**
- Lower overall volume
- Includes notable Native American representation
- Could indicate patterns from tribal or rural areas with different reporting styles or issues

#### Assign patterns to reflect a demographically distinct crime profile
- Pattern 0 --> Low-volume youth crime
- Pattern 1 --> High Volume mixed-race youth crime
- Pattern 2 --> Suburban White-dominated high youth crime
- Patttern 3 --> Urban Black youth crime
- Pattern 4 --> Low-volume rural/tribal youth crime


#### Transition Modeling - How Patterns Evolve Over Time
Build a transition matrix that shows how states move from one pattern to another over the years.

In [50]:
transitions = defaultdict(lambda: defaultdict(int))

# loop over each state to collect pattern transitions year over year
for state in model_df['State'].unique():
    state_data = model_df[model_df['State'] == state].sort_values('Year')
    patterns = state_data['Pattern'].tolist()

    for i in range(len(patterns) - 1):
        from_pattern = patterns[i]
        to_pattern = patterns[i+1]
        transitions[from_pattern][to_pattern] += 1

# covert to a probability matrix
# normalize transition counts to probabilities 
transition_matrix = {
    from_p: {
        to_p: count / sum(to_counts.values())
        for to_p, count in to_counts.items()
    }
    for from_p, to_counts in transitions.items()
}

# create a full matrix of all posivle transitions
patterns = sorted(model_df['Pattern'].unique())
matrix_df = pd.DataFrame(index=patterns, columns=patterns).fillna(0.0)

for from_p, to_dict in transition_matrix.items():
    for to_p, prob in to_dict.items():
        matrix_df.loc[from_p, to_p] = prob

matrix_df.index.name = 'From'
matrix_df.columns.name = 'To'

matrix_df.round(3)

To,0,1,2,3,4
From,,,,,
0,0.871,0.000,0.000,0.085,0.044
1,0.000,0.831,0.015,0.154,0.000
2,0.000,0.231,0.769,0.000,0.000
3,0.273,0.019,0.000,0.681,0.027
4,0.652,0.000,0.000,0.146,0.202


#### Analysis of Transitio Probability Matrix

#### Pattern 0
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 0 | 87% |
| Moves to Pattern 3  | 9% |
| Moves to Pattern 4 | 4% |

**Interpretation:**
- Highly stable pattern — most states stay here.
- Small chance of transitioning to Patterns 3 or 4.
- Likely represents a baseline or low-risk demographic crime pattern.

#### Pattern 1
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 1 | 83% |
| Moves to Pattern 2  | 2% |
| Moves to Pattern 3 | 15% |

**Interpretation:**
- Also fairly stable.
- But interestingly, 15% of transitions go to Pattern 3, possibly suggesting escalation.
- Could represent youth-heavy or racially mixed crime patterns shifting toward higher intensity.

#### Pattern 2
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 2 | 77% |
| Moves to Pattern 1  | 23% |

**Interpretation:**
- Somewhat stable, but with a substantial chance of returning to Pattern 1.
- Suggests an oscillating behavior between two patterns — possibly linked to policy shifts or local enforcement.

#### Pattern 3
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 3 | 68% |
| Moves to Pattern 0  | 27% |

**Interpretation:**
- Pattern 3 is less stable, often falling back to Pattern 0.
- May represent transient crime types that resolve or deescalate.

#### Pattern 4
| *Transition* | *Probability* |
| --- | --- |
| Stays in Pattern 4 | 20% |
| Moves to Pattern 0  | 65% |
| Moves to Pattern 3 | 15% |

**Interpretation:**
- Very unstable pattern — 80% of the time it transitions away.
- Most often into Pattern 0, possibly indicating dissipation of crime risk.
- May represent special, high-intensity or isolated crime patterns (e.g., tribal regions or rare cases).

#### Overal Insights on the Pattern Transitions
- Patterns 0 and 1 are “attractors” — states tend to settle into them.
- Pattern 4 is volatile and quickly disperses — could be a high-risk but temporary scenario.
- Frequent returns to Pattern 0 suggest it might be a “default” or low-risk state.

### Risk Scoring & Forecasting
Assign a risk score to each predicted pattern for each state in a future year, then visualize the risk across the U.S.

In [58]:
pattern_risks = {
    0: 0.2, # low volume, mixed demographics
    1: 0.7, # high volume, Black/White balances
    2: 0.1, # extreme volume, White dominated
    3: 0.8, # mid volume, Black dominated
    4: 0.5, # low volume but unstable, rural 
}

def forecast_pattern(state, start_year, num_years, df, matrix):
    forecasted_patterns = {}
    
    # get initial patterns from the last real year
    row = df[(df['State'] == state) & (df['Year'] == start_year)]
    if row.empty:
        return forecasted_patterns
    
    current_pattern = row['Pattern'].values[0]
    
    for i in range(1, num_years + 1):
        next_year = start_year + i
        transitions = matrix.get(current_pattern, {})
        if transitions:
            next_pattern = max(transitions.items(), key=lambda x: x[1])[0]
        else:
            next_pattern = current_pattern
        
        forecasted_patterns[next_year] = next_pattern
        current_pattern = next_pattern
    
    return forecasted_patterns

start_year = 2020
num_years = 3

multi_year_forecast = []

for state in model_df['State'].unique():
    pattern_sequence = forecast_pattern(
        state,
        start_year=start_year,
        num_years=num_years,
        df=model_df,
        matrix=transition_matrix
    )

    for year, pattern in pattern_sequence.items():
        risk = pattern_risks.get(pattern, 0.0)
        multi_year_forecast.append({
            'State': state,
            'Year': year,
            'Predicted Pattern': pattern,
            'Risk Score': risk
        })

multi_forecast_df = pd.DataFrame(multi_year_forecast)
multi_forecast_df


,State,Year,Predicted Pattern,Risk Score
0,AK,2021,0,0.2
1,AK,2022,0,0.2
2,AK,2023,0,0.2
3,AR,2021,0,0.2
4,AR,2022,0,0.2
...,...,...,...,...
139,WV,2022,0,0.2
140,WV,2023,0,0.2
141,WY,2021,0,0.2
142,WY,2022,0,0.2


In [59]:
fig = px.choropleth(
    multi_forecast_df,
    locations='State',
    locationmode='USA-states',
    color='Risk Score',
    scope='usa',
    animation_frame='Year',
    color_continuous_scale='Reds',
    range_color=(0, 1),
    labels={'Risk Score': 'Crime Risk'},
    title="Forecasted Crime Risk by State (Animated Over Years)"
)

fig.update_layout(
    geo=dict(lakecolor='rgb(255, 255, 255)'),
    margin=dict(l=20, r=20, t=50, b=20),
    coloraxis_colorbar=dict(
        title="Risk Score",
        ticks="outside"
    )
)

fig.show()